# Task 2: Embeddings and Semantic Search

This task implements FAQ search using embeddings,
cosine similarity, semantic search and keyword search.

In [3]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

print("Libraries imported successfully!")

Libraries imported successfully!


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\YOHO_LAP_22\Desktop\Week3_Weekly_Assessment\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\YOHO_LAP_22\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (6 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [5]:
# Create 12 FAQ records

faqs = [
    # HR FAQs
    {
        "category": "HR",
        "question": "What are the work from home rules?",
        "answer": "Employees can work remotely up to two days per week with manager approval."
    },
    {
        "category": "HR",
        "question": "How many annual leave days do employees get?",
        "answer": "Employees receive 20 annual leave days per year."
    },
    {
        "category": "HR",
        "question": "How can I apply for sick leave?",
        "answer": "Submit a sick leave request through the HR portal."
    },
    {
        "category": "HR",
        "question": "Who should I contact for workplace complaints?",
        "answer": "Contact the HR department for workplace complaints."
    },

    # IT FAQs
    {
        "category": "IT",
        "question": "How can I reset my login password?",
        "answer": "Use the forgot password option on the company login page."
    },
    {
        "category": "IT",
        "question": "What should I do if I cannot log in?",
        "answer": "Check your internet connection and contact IT support if the issue continues."
    },
    {
        "category": "IT",
        "question": "How do I request a new company laptop?",
        "answer": "Submit a hardware request through the IT service desk."
    },
    {
        "category": "IT",
        "question": "How can I connect to the company VPN?",
        "answer": "Install the approved VPN application and sign in using your company credentials."
    },

    # Payroll FAQs
    {
        "category": "Payroll",
        "question": "When is the monthly salary credited?",
        "answer": "Monthly salary is credited on the last working day of each month."
    },
    {
        "category": "Payroll",
        "question": "How can I download my payslip?",
        "answer": "Log in to the employee payroll portal and download your payslip."
    },
    {
        "category": "Payroll",
        "question": "Why is my salary lower this month?",
        "answer": "Salary differences may result from tax deductions, unpaid leave or other adjustments. Contact payroll for details."
    },
    {
        "category": "Payroll",
        "question": "How can I update my bank account details?",
        "answer": "Submit a bank account update request through the payroll department."
    }
]

# Convert FAQs into a DataFrame
faq_df = pd.DataFrame(faqs)

print("Total FAQs:", len(faq_df))
print("\nFAQ category distribution:")
print(faq_df["category"].value_counts())

display(faq_df)

Total FAQs: 12

FAQ category distribution:
category
HR         4
IT         4
Payroll    4
Name: count, dtype: int64


,category,question,answer
0,HR,What are the work from home rules?,Employees can work remotely up to two days per...
1,HR,How many annual leave days do employees get?,Employees receive 20 annual leave days per year.
2,HR,How can I apply for sick leave?,Submit a sick leave request through the HR por...
3,HR,Who should I contact for workplace complaints?,Contact the HR department for workplace compla...
4,IT,How can I reset my login password?,Use the forgot password option on the company ...
5,IT,What should I do if I cannot log in?,Check your internet connection and contact IT ...
6,IT,How do I request a new company laptop?,Submit a hardware request through the IT servi...
7,IT,How can I connect to the company VPN?,Install the approved VPN application and sign ...
8,Payroll,When is the monthly salary credited?,Monthly salary is credited on the last working...
9,Payroll,How can I download my payslip?,Log in to the employee payroll portal and down...


In [6]:
# Generate embeddings for all FAQ questions

faq_questions = faq_df["question"].tolist()

faq_embeddings = model.encode(
    faq_questions,
    convert_to_numpy=True
)

print("Embeddings generated successfully!")
print("Embedding shape:", faq_embeddings.shape)


Embeddings generated successfully!
Embedding shape: (12, 384)


In [7]:
faq_df.to_csv("faqs.csv", index=False)

np.save("faq_embeddings.npy", faq_embeddings)

print("FAQ dataset and embeddings saved successfully!")

FAQ dataset and embeddings saved successfully!


In [8]:
def cosine_similarity(a, b):
    """
    Calculate cosine similarity between two vectors using NumPy.
    """

    dot_product = np.dot(a, b)

    magnitude_a = np.linalg.norm(a)
    magnitude_b = np.linalg.norm(b)

    if magnitude_a == 0 or magnitude_b == 0:
        return 0.0

    similarity = dot_product / (magnitude_a * magnitude_b)

    return float(similarity)


print("Cosine similarity function created!")

Cosine similarity function created!


In [9]:
# Compare the first two FAQ embeddings

similarity_score = cosine_similarity(
    faq_embeddings[0],
    faq_embeddings[1]
)

print("Cosine similarity:", round(similarity_score, 4))

Cosine similarity: 0.2853


In [10]:
def semantic_search(query, top_k=3):

    # Convert user query into an embedding
    query_embedding = model.encode(
        query,
        convert_to_numpy=True
    )

    # Calculate similarity against all FAQ embeddings
    scores = []

    for embedding in faq_embeddings:
        score = cosine_similarity(query_embedding, embedding)
        scores.append(score)

    # Get top matching FAQ indexes
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for index in top_indices:
        results.append({
            "Category": faq_df.iloc[index]["category"],
            "Question": faq_df.iloc[index]["question"],
            "Answer": faq_df.iloc[index]["answer"],
            "Similarity": round(scores[index], 4)
        })

    return pd.DataFrame(results)


print("Semantic search function created!")

Semantic search function created!


In [11]:
result = semantic_search("I cannot log in")

display(result)

,Category,Question,Answer,Similarity
0,IT,What should I do if I cannot log in?,Check your internet connection and contact IT ...,0.8641
1,IT,How can I reset my login password?,Use the forgot password option on the company ...,0.6085
2,Payroll,How can I update my bank account details?,Submit a bank account update request through t...,0.2464


In [12]:
test_queries = [
    "I cannot log in",
    "when is salary credited",
    "work from home rules"
]

for query in test_queries:

    print("\nQUERY:", query)
    print("-" * 60)

    display(semantic_search(query, top_k=3))


QUERY: I cannot log in
------------------------------------------------------------


,Category,Question,Answer,Similarity
0,IT,What should I do if I cannot log in?,Check your internet connection and contact IT ...,0.8641
1,IT,How can I reset my login password?,Use the forgot password option on the company ...,0.6085
2,Payroll,How can I update my bank account details?,Submit a bank account update request through t...,0.2464



QUERY: when is salary credited
------------------------------------------------------------


,Category,Question,Answer,Similarity
0,Payroll,When is the monthly salary credited?,Monthly salary is credited on the last working...,0.8080
1,Payroll,Why is my salary lower this month?,Salary differences may result from tax deducti...,0.3645
2,Payroll,How can I download my payslip?,Log in to the employee payroll portal and down...,0.3142



QUERY: work from home rules
------------------------------------------------------------


,Category,Question,Answer,Similarity
0,HR,What are the work from home rules?,Employees can work remotely up to two days per...,0.8798
1,HR,How many annual leave days do employees get?,Employees receive 20 annual leave days per year.,0.2705
2,HR,Who should I contact for workplace complaints?,Contact the HR department for workplace compla...,0.2302


In [13]:
def keyword_search(query):

    query_words = set(query.lower().split())

    scores = []

    for question in faq_df["question"]:

        question_words = set(question.lower().split())

        # Count matching words
        matching_words = query_words.intersection(question_words)

        scores.append(len(matching_words))

    # Find best matching FAQ
    best_index = np.argmax(scores)

    if scores[best_index] == 0:
        return pd.DataFrame()

    return pd.DataFrame([{
        "Category": faq_df.iloc[best_index]["category"],
        "Question": faq_df.iloc[best_index]["question"],
        "Answer": faq_df.iloc[best_index]["answer"],
        "Keyword Matches": scores[best_index]
    }])


print("Keyword search function created successfully!")

Keyword search function created successfully!


In [14]:
result = keyword_search("I cannot log in")

display(result)

,Category,Question,Answer,Keyword Matches
0,IT,What should I do if I cannot log in?,Check your internet connection and contact IT ...,3


In [15]:
test_queries = [
    "I cannot log in",
    "when is salary credited",
    "work from home rules"
]

comparison_results = []

for query in test_queries:

    keyword_result = keyword_search(query)
    semantic_result = semantic_search(query, top_k=1)

    keyword_question = (
        keyword_result.iloc[0]["Question"]
        if not keyword_result.empty else "No match"
    )

    semantic_question = (
        semantic_result.iloc[0]["Question"]
        if not semantic_result.empty else "No match"
    )

    comparison_results.append({
        "Query": query,
        "Keyword Result": keyword_question,
        "Semantic Result": semantic_question
    })

comparison_df = pd.DataFrame(comparison_results)

display(comparison_df)

,Query,Keyword Result,Semantic Result
0,I cannot log in,What should I do if I cannot log in?,What should I do if I cannot log in?
1,when is salary credited,When is the monthly salary credited?,When is the monthly salary credited?
2,work from home rules,What are the work from home rules?,What are the work from home rules?


In [16]:
semantic_test_queries = [
    "My credentials are not being accepted",
    "On which day will my wages arrive?"
]

for query in semantic_test_queries:

    print("\nQUERY:", query)

    print("\nKeyword Search:")
    display(keyword_search(query))

    print("\nSemantic Search:")
    display(semantic_search(query, top_k=3))


QUERY: My credentials are not being accepted

Keyword Search:


,Category,Question,Answer,Keyword Matches
0,HR,What are the work from home rules?,Employees can work remotely up to two days per...,1



Semantic Search:


,Category,Question,Answer,Similarity
0,IT,What should I do if I cannot log in?,Check your internet connection and contact IT ...,0.5128
1,IT,How can I reset my login password?,Use the forgot password option on the company ...,0.4652
2,Payroll,How can I update my bank account details?,Submit a bank account update request through t...,0.2800



QUERY: On which day will my wages arrive?

Keyword Search:


,Category,Question,Answer,Keyword Matches
0,IT,How can I reset my login password?,Use the forgot password option on the company ...,1



Semantic Search:


,Category,Question,Answer,Similarity
0,Payroll,When is the monthly salary credited?,Monthly salary is credited on the last working...,0.5411
1,Payroll,Why is my salary lower this month?,Salary differences may result from tax deducti...,0.4808
2,HR,How many annual leave days do employees get?,Employees receive 20 annual leave days per year.,0.4055


## Keyword Search vs Semantic Search

1. Keyword search identifies FAQs using exact matching words.
2. Semantic search understands the meaning and context of a query using embeddings and cosine similarity.
3. Semantic search can retrieve relevant FAQs even when users use different words from the original FAQ questions.

### Two Examples Where Semantic Search Can Win

1. Query: "My credentials are not being accepted"
   - Relevant FAQ: What should I do if I cannot log in?
   - Reason: The meaning is similar even though the wording differs.

2. Query: "On which day will my wages arrive?"
   - Relevant FAQ: When is the monthly salary credited?
   - Reason: Semantic search can identify the relationship between wages arriving and salary being credited.

In [17]:
comparison_df.to_csv(
    "search_comparison.csv",
    index=False
)

print("Comparison table saved successfully!")

Comparison table saved successfully!
